# Chapter 4 (part 4) — Regression Diagnostics

Covers: Outliers → Influential Values → Heteroskedasticity, Non-Normality, and Correlated Errors → Partial Residual Plots and Nonlinearity, on Ames Housing. (Non-normality is covered in the exam only.)

The thread: **every regression comes with promises (linear mean, independent errors, constant spread, ...), and diagnostics tell you which promises broke and what each break costs.** Some violations bias the coefficients, some only wreck the standard errors, some only matter for prediction intervals. Knowing which is which is the senior-level intuition.

Base model for the exercises: `Sale_Price ~ Gr_Liv_Area + Year_Built + Garage_Area + Total_Bsmt_SF` (fit in the setup cell as `base`). The exam uses a *different* model.

**How to work each section** (~10 min each; ~1 hour total with the exam): write your one-line prediction in the *Prediction* cell **before** running anything, do the code, then write 2-3 bullets on where the result matched or broke your prediction and why. Ask for a hint if you get stuck.

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.outliers_influence import OLSInfluence
from statsmodels.stats.stattools import durbin_watson
from statsmodels.stats.diagnostic import het_breuschpagan

pd.set_option("display.max_columns", None)

from mlstats.data import load_ames
ames = load_ames()
base = smf.ols('Sale_Price ~ Gr_Liv_Area + Year_Built + Garage_Area + Total_Bsmt_SF', ames).fit()
base.summary().tables[1]

## 1. Outliers and Influential Values

An **outlier** is a house whose price is far from the model's prediction. We standardize residuals so "3" means "three typical misses": if typical misses are about $41k, a house missed by $123k scores 3. (*Studentized* residuals refit without that point first, so it can't hide by pulling the line toward itself.)

A point has **influence** if deleting it changes the coefficients. That needs *both* a big miss and **leverage** (being far from the other houses in x-space). Numeric picture: balance a ruler on a pencil. A weight near the middle barely tilts it; a weight at the far end tips it hard, unless it sits right on the line the ruler already follows. **Cook's distance** combines both: roughly "how far all the fitted values move if this point is deleted".

**Predict first:** will dropping just the 5 most influential of 2,930 homes move any coefficient by more than 1 standard error?

**Question 1.** (a) How many homes have |studentized residual| > 3? Compare that fraction to what a normal distribution would give. (b) Refit without the 5 highest-Cook's-D homes. How far does each coefficient move, in units of its standard error? (c) Print the homes with `Gr_Liv_Area > 4000`, including `Sale_Condition`. Would you remove them? Give one justification you'd defend and one you wouldn't.

*Tools for this question:*
- `infl = OLSInfluence(res)`; `infl.resid_studentized_external` is the studentized-residual array. Count with `(np.abs(x) > 3).sum()`.
- `infl.cooks_distance` returns a tuple `(distances, p_values)`; take `[0]`. `df.drop(index=idx_list)` to refit without rows.

_Prediction (one line, before running anything):_

_What matched, what surprised you, and why (2-3 bullets):_

## 2. Heteroskedasticity

OLS assumes the spread of errors is the same at every x. With houses that's rarely true: a $100k house might miss by $15k, a $600k house by $80k. Uneven spread leaves the coefficients **unbiased**, but the standard-error formula assumes the wrong variance, so p-values and confidence intervals are off, in either direction.

**Predict first:** will HC3 robust standard errors be bigger or smaller than the classical ones? Will the coefficients change?

**Question 2.** (a) Plot residuals against fitted values and compute the residual SD within fitted-value quintiles. Run `het_breuschpagan`. With n = 2,930, what does a tiny p-value really tell you? (b) Refit with `HC3` standard errors and compare coefficients and SEs. (c) Refit on `np.log(Sale_Price)` and recheck the spread by quintile. Which fix would you pick for "what's a year of age worth?" vs. "predict dollars for a new listing"?

*Tools for this question:*
- `het_breuschpagan(res.resid, res.model.exog)` (from `statsmodels.stats.diagnostic`) returns `(LM stat, LM p-value, F stat, F p-value)`.
- Robust SEs: `smf.ols(...).fit(cov_type="HC3")`. Quintiles: `pd.qcut(res.fittedvalues, 5, labels=False)` then `groupby(...)["resid"].std()`.

_Prediction (one line, before running anything):_

_What matched, what surprised you, and why (2-3 bullets):_

## 3. Correlated Errors

If one house's error tells you something about its neighbor's, you have less independent information than n suggests, so standard errors are too small. The **Durbin-Watson** statistic checks consecutive errors: 2 means no correlation, near 0 strong positive, near 4 negative. Numeric picture: if each residual is mostly a copy of the previous one, you get long runs of same-sign residuals and DW near 0. DW depends entirely on the row order you give it, and this file is sorted roughly by neighborhood, not by date.

**Predict first:** will DW in the file's row order be close to 2 or well below 2? In time order?

**Question 3.** (a) Compute DW for the base model's residuals in file order, in time order (`Year_Sold`, `Mo_Sold`, random tiebreak), and in `Neighborhood` order. Why do they differ, and which one answers a question about *time*? (b) Refit with standard errors clustered by neighborhood. How do the SEs compare, and what's the caveat with only 28 clusters?

*Tools for this question:*
- `durbin_watson(resid_array)` from `statsmodels.stats.stattools`; order the residuals first: `res.resid.loc[order_index]`. Random tiebreak: `df.sample(frac=1, random_state=0).sort_values(["Year_Sold", "Mo_Sold"], kind="stable").index`.
- Clustered SEs: `.fit(cov_type="cluster", cov_kwds={"groups": codes})` with `codes = pd.factorize(df["Neighborhood"])[0]`.

_Prediction (one line, before running anything):_

_What matched, what surprised you, and why (2-3 bullets):_

## 4. Partial Residual Plots and Nonlinearity

A residual-vs-fitted plot can look fine even when one predictor has a curved relationship, because the other linear terms mask it. A **partial residual plot** isolates one predictor j: plot `residual + coef_j * x_j` against `x_j`. If the relationship is linear, the points scatter around a straight line with slope coef_j. Numeric picture: if the truth is a hockey stick (flat, then steep) and you fit a line, the points sit *above* the line at both ends and *below* it in the middle, a U shape.

**Predict first:** which of the four predictors will curve, and which way (U, upside-down U, hockey stick)?

**Question 4.** (a) Make partial residual plots for all four predictors. Which are well described by a line, and which curve? (b) Add `I(Year_Built**2)` and compare R², RMSE, and the new partial residual plot for `Year_Built`. Did the curve go away?

*Tools for this question:*
- `sm.graphics.plot_ccpr(res, "Year_Built")` (one predictor) or `sm.graphics.plot_ccpr_grid(res)` (all).
- Quadratic term: `I(Year_Built**2)` in the formula.

_Prediction (one line, before running anything):_

_What matched, what surprised you, and why (2-3 bullets):_

---
# Exam — check your intuition

5 questions, about 3-4 minutes each. Almost none need code: the model outputs are printed for you as text, and you're asked to *read*, *predict*, or *explain*. Scratch code is fine if you want to poke at something, but try to answer from understanding first.

How it works:
- Write your answer in the **Your answer** cell under each question. Plain words, 2-3 bullets — the way you'd say it out loud in an interview.
- Leave the **Review** cell alone. When you're done, ask Claude to review; it fills those cells in. Being right is a perfectly valid review outcome.
- Each question is tagged with the role it leans toward — **DS** (general data science), **Causal** (decision science / causal inference), **ML** (machine learning). Anything tagged for one role is still fair game for the others.
- Several questions are phrased the way interviewers ask them. A good answer is short, names the mechanism, and says what you'd *do* about it.

### Exam Q1 — Outlier vs. leverage vs. influence
*Role lean: DS · interview*

Four houses from Model E (`Sale_Price ~ Gr_Liv_Area + Year_Built + Lot_Area`), with the usual diagnostics:

```text
Model E: Sale_Price~Gr_Liv_Area+Year_Built+Lot_Area   (n = 2930, p = 4 parameters)
Rule-of-thumb flags: |studentized residual| > 3;  leverage > 2p/n = 0.0027;  Cook's D > 4/n = 0.0014

   Gr_Liv_Area  Lot_Area  Year_Built  Sale_Price  stud_resid  leverage_h  cooks_D
A         5642     63887        2008      160000       -11.0      0.0308    0.927
B         2470     12720        2003      615000         6.7      0.0017    0.019
C         2036    215245        1965      375000        -0.9      0.2470    0.072
D         1686     10240        1980      207500         0.0      0.0004    0.000
```

(a) For each of A-D say whether it is an *outlier* (unusual y given x), a *high-leverage point* (unusual x), both, or neither. (b) Which one is truly *influential*, i.e., would change the fitted coefficients noticeably if deleted? Explain why high leverage alone or a large residual alone is not enough. (c) In which direction would you expect the fitted line to be dragged by house A, and what does that do to the `Lot_Area` coefficient? (d) Interview version: "A single data point can swing your regression. Why, and how do you detect it?" 

**Your answer (Q1):**

> _write here_

**Review (Q1):**

> _pending — filled in after you answer_

### Exam Q2 — Reading a heteroskedasticity diagnosis
*Role lean: DS · Causal · ML*

Same model. Look at how the typical miss changes with the predicted price, and at two ways of computing standard errors:

```text
Residual spread by fitted-value quintile (cheapest predicted homes -> most expensive):
    avg_fitted_price  resid_sd  resid_sd_in_log_model
fq                                                   
0              98431     22441                   0.26
1             138304     24072                   0.19
2             176945     31100                   0.20
3             215836     40991                   0.18
4             274464     78905                   0.27

Coefficient standard errors, same model:
                 coef  classical SE  HC3 robust SE
Gr_Liv_Area    91.685         1.815          4.389
Year_Built   1098.945        29.087         28.949
Lot_Area        0.924         0.113          0.226
```

(a) Describe the pattern in plain words. What would the residuals-versus-fitted plot look like (draw the shape)? (b) Which parts of the output does this problem make *wrong*: the coefficients, their standard errors, the R-squared, the predictions? Be specific about what stays unbiased and what doesn't. (c) The `Gr_Liv_Area` standard error more than doubles with the robust formula while `Year_Built` barely moves. What does that tell you about where in the data the heteroskedasticity is concentrated? (d) Name two fixes and what each buys you (hint: one changes the *model*, one changes only the *standard errors*). (e) After taking logs, the residual spread is roughly even across fitted values. What did the log transform change about the *meaning* of the coefficients?

**Your answer (Q2):**

> _write here_

**Review (Q2):**

> _pending — filled in after you answer_

### Exam Q3 — Non-normal residuals: what actually breaks?
*Role lean: DS · ML · interview*

```text
Residual skewness = 0.53   excess kurtosis = 12.2   (a normal distribution has 0 and 0)

95% prediction intervals from the classical formula, fit on 2,051 training homes, checked on 879 held-out homes:
  held-out homes whose true price fell inside their interval, overall:  96.0%
  ... among the cheapest third:  99.7%
  ... among the middle third:  99.3%
  ... among the priciest third:  89.1%
```

(a) The overall coverage is 96% — close to the 95% target — but that average hides something. Explain the pattern across price tiers using what you know about the residual spread from the previous question. (b) A reviewer says "your residuals aren't normal, so none of your p-values are valid." With n = 2,930, how much of that is true for the *coefficient* p-values, and how much for the *prediction intervals*? (Hint: think about what the CLT does for one but not the other.) (c) What would you do to produce honest prediction intervals here (name two approaches)? (d) If the business question were "give me a range I can promise a seller", which diagnostic from this exam would you care about most?

**Your answer (Q3):**

> _write here_

**Review (Q3):**

> _pending — filled in after you answer_

### Exam Q4 — Is the independence assumption satisfied?
*Role lean: Causal · DS*

```text
Durbin-Watson of the residuals of Model E (2.0 = no first-order correlation; <2 = neighbors in the ordering resemble each other):
  rows in the order they appear in the file : 1.33
  rows sorted by sale date (ties within a month in random order): 1.95
  rows sorted by Neighborhood               : 1.26

Average residual (actual - fitted) for a few neighborhoods:
                    avg_residual  homes
Neighborhood                           
Landmark                -45065.0      1
Meadow_Village          -38885.0     37
Briardale               -32784.0     30
Northridge_Heights       62015.0    166
Stone_Brook              70223.0     51
Green_Hills              86855.0      2

Standard error of Gr_Liv_Area: classical 1.8   HC3 robust 4.4   clustered by neighborhood (28 clusters) 12.1
```

(a) Why does the Durbin-Watson statistic give such different answers depending on how you sort the rows? What does each of the three numbers tell you about what the data *looks* like (a time effect, or something else)? (b) Based on the neighborhood residual averages, what is the real violation of independence in this dataset, and what does it do to the standard errors? Does it bias the coefficients? (c) The clustered SE for Gr_Liv_Area is much larger than the others. Why is this the "honest" one for a causal question, and what caveat applies when you only have 28 clusters? (d) How would you change the model (not the SEs) to absorb some of this?

**Your answer (Q4):**

> _write here_

**Review (Q4):**

> _pending — filled in after you answer_

### Exam Q5 — Which assumption protects which claim?
*Role lean: Causal · DS · ML · interview*

Interviewers love a version of this: build a 4-row table in your answer, one row per claim, and for each say which regression assumptions it *needs* and which it *doesn't*. The claims: (1) the coefficient estimates are **unbiased** for the true linear relationship; (2) the **standard errors and p-values** are valid; (3) a **prediction interval** for a new house has the stated coverage; (4) the **point predictions** are good on average. The assumptions: linearity of the mean, errors independent, errors with constant variance, errors normal, no omitted confounders (for a causal reading), no perfect multicollinearity. Then add one sentence on which assumption failures you'd worry about *least* when the only goal is point prediction.

**Your answer (Q5):**

> _write here_

**Review (Q5):**

> _pending — filled in after you answer_